In [8]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import scipy.io as io

In [9]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
# cuda
print("Using device:", device)

Using device: mps


#### Data


In [12]:
interictal_1 = '//Users/poorvaichandrasen/Downloads/Patient_1/Patient_1_interictal_segment_0001.mat'
i_eg_1 = io.loadmat(interictal_1)
interictal_segment_1 = i_eg_1.get('interictal_segment_1')

In [14]:
my_data= interictal_segment_1[0][0][0]

In [16]:
data_np = my_data[:, 0:100000]
type(data_np)

numpy.ndarray

In [18]:
SEQ_LEN = 50
data = data_np.copy().astype(np.float32)  # shape (15, 5000)

# Normalize each channel independently
scaler = StandardScaler()
for i in range(data.shape[0]):
    data[i] = scaler.fit_transform(data[i].reshape(-1, 1)).flatten()

In [20]:
print(data_np.shape)

(15, 100000)


In [22]:
# Create (X, y) pairs for next-timestep prediction
X, y = [], []
for ch in data:
    for i in range(len(ch) - SEQ_LEN):
        X.append(ch[i:i+SEQ_LEN])
        y.append(ch[i+SEQ_LEN])



In [24]:
X = np.array(X).reshape(-1, 50, 1)
y = np.array(y).reshape(-1, 1)

In [25]:
np.random.seed(0)
n = len(X)
indices = np.random.permutation(n)
train_idx = int(0.7 * n)
val_idx = int(0.85 * n)

In [28]:
train_x, train_y = X[indices[:train_idx]], y[indices[:train_idx]]
val_x, val_y = X[indices[train_idx:val_idx]], y[indices[train_idx:val_idx]]
test_x, test_y = X[indices[val_idx:]], y[indices[val_idx:]]

In [30]:
train_ds = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
val_ds = TensorDataset(torch.from_numpy(val_x), torch.from_numpy(val_y))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

#### Model

In [32]:
class EEG_RNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=5, output_size=1):
        super(EEG_RNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Create h0 on the same device as the input tensor x
        h0 = torch.zeros(1, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])  # last timestep
        return out

#### Training


In [42]:
model = EEG_RNN().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 200
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            val_loss += loss.item() * len(xb)

    if epoch%10 == 0:
        print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss/len(train_ds):.6f} | Val Loss: {val_loss/len(val_ds):.6f}")
        print("Using device:", device)


Epoch 01 | Train Loss: 0.015095 | Val Loss: 0.000294
Using device: mps


KeyboardInterrupt: 

#### Using

In [ ]:
path = '/Users/poorvaichandrasen/GCN_IIST/EEG/my_RNN_model.pth'
torch.save(model.state_dict(), path)

In [40]:
model = EEG_RNN()  # must match the original model architecture
#model.load_state_dict(torch.load(path))
model.eval() 

EEG_RNN(
  (rnn): RNN(1, 5, batch_first=True)
  (fc): Linear(in_features=5, out_features=1, bias=True)
)

In [30]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def evaluate_model(model, test_x, test_y):
    model.eval()  # Set model to eval mode (important for dropout/BATCHNORM if used)

    with torch.no_grad():
        predictions = model(test_x).squeeze().cpu().numpy()
        targets = test_y.squeeze().cpu().numpy()
        mse = mean_squared_error(targets, predictions)
        mae = mean_absolute_error(targets, predictions)
        r2 = r2_score(targets, predictions)
        
        print("📊 Evaluation Metrics:")
        print(f"✅ Mean Squared Error (MSE):      {mse:.6f}")
        print(f"✅ Mean Absolute Error (MAE):     {mae:.6f}")
        print(f"✅ R² Score (Explained Variance): {r2:.4f}")

        for i in range(20):
         print('Predicted:',predictions[i], 'Target:', targets[i], 'loss =', targets[i]-predictions[i])

In [44]:
evaluate_model(model, torch.from_numpy(test_x), torch.from_numpy(test_y))

📊 Evaluation Metrics:
✅ Mean Squared Error (MSE):      1.249771
✅ Mean Absolute Error (MAE):     0.936055
✅ R² Score (Explained Variance): -0.3131
Predicted: 0.6195308 Target: -0.4093434 loss = -1.0288742
Predicted: 0.59568787 Target: -1.0351912 loss = -1.630879
Predicted: 0.6364509 Target: 0.041457865 loss = -0.594993
Predicted: 0.6172296 Target: -0.4597423 loss = -1.0769719
Predicted: 0.64699256 Target: 0.2513258 loss = -0.39566678
Predicted: 0.6291212 Target: -0.17799707 loss = -0.80711824
Predicted: 0.58711225 Target: -1.2426126 loss = -1.8297248
Predicted: 0.587641 Target: -1.2289023 loss = -1.8165433
Predicted: 0.587843 Target: -1.2043756 loss = -1.7922187
Predicted: 0.5859302 Target: -1.3790705 loss = -1.9650007
Predicted: 0.64073336 Target: 0.09992924 loss = -0.54080415
Predicted: 0.64857674 Target: 0.25544626 loss = -0.39313048
Predicted: 0.6150067 Target: -0.5162713 loss = -1.131278
Predicted: 0.58951896 Target: -1.1743916 loss = -1.7639105
Predicted: 0.66314685 Target: 0.557

In [357]:
data_t = my_data[0:15, 280000:280200]
data_t.shape

(15, 200)

In [359]:
SEQ_LEN = 50
data_tf = data_t.copy().astype(np.float32)  # shape (15, 5000)

# Normalize each channel independently
scaler = StandardScaler()
for i in range(data_tf.shape[0]):
    data_tf[i] = scaler.fit_transform(data_tf[i].reshape(-1, 1)).flatten()

In [362]:
# Create (X, y) pairs for next-timestep prediction
X_tf, y_tf = [], []
for ch in data_tf:
    for i in range(len(ch) - SEQ_LEN):
        X_tf.append(ch[i:i+SEQ_LEN])
        y_tf.append(ch[i+SEQ_LEN])


In [364]:
X_tf = np.array(X_tf).reshape(-1, 50, 1)
y_tf = np.array(y_tf).reshape(-1, 1)

In [376]:
evaluate_model(model, torch.from_numpy(X_tf), torch.from_numpy(y_tf))

📊 Evaluation Metrics:
✅ Mean Squared Error (MSE):      0.002665
✅ Mean Absolute Error (MAE):     0.038005
✅ R² Score (Explained Variance): 0.9969
Predicted: 1.0912632 Target: 1.0953456 loss = 0.0040824413
Predicted: 1.0852056 Target: 1.0953456 loss = 0.010140061
Predicted: 1.1008711 Target: 1.0953456 loss = -0.00552547
Predicted: 1.105702 Target: 1.1330447 loss = 0.027342677
Predicted: 1.1500899 Target: 1.1330447 loss = -0.01704514
Predicted: 1.1234497 Target: 1.0953456 loss = -0.028104067
Predicted: 1.058627 Target: 1.1330447 loss = 0.07441771
Predicted: 1.1551 Target: 1.1707437 loss = 0.015643716
Predicted: 1.2109997 Target: 1.2084428 loss = -0.00255692
Predicted: 1.2170774 Target: 1.2461418 loss = 0.029064417
Predicted: 1.2418185 Target: 1.1707437 loss = -0.07107484
Predicted: 1.1178551 Target: 1.0953456 loss = -0.022509456
Predicted: 1.0511765 Target: 1.0576466 loss = 0.006470084
Predicted: 1.0603628 Target: 1.0576466 loss = -0.0027161837
Predicted: 1.0725427 Target: 1.0953456 loss

#### Obtaining the embeddings

In [50]:
def extract_channel_embeddings(model, data, seq_len=50, device='cpu'):
    """
    Returns a tensor of shape [15, hidden_dim] — one embedding per channel
    """
    model.eval()
    model.to(device)

    eeg_data = data.astype(np.float32)
    n_channels, n_time = eeg_data.shape
    embeddings = []

    with torch.no_grad():
        for ch in range(n_channels):
            signal = eeg_data[ch]
            ch_embeddings = []

            # Slide over the channel time series
            for i in range(n_time - seq_len):
                window = signal[i:i+seq_len]
                window_tensor = torch.tensor(window, dtype=torch.float32).view(1, seq_len, 1).to(device)

                # Get hidden state
                output, hn = model.rnn(window_tensor)  # hn: [1, 1, hidden_dim]
                final_hidden = hn.squeeze(0).squeeze(0)  # shape: [hidden_dim]
                ch_embeddings.append(final_hidden.cpu().numpy())

            # Pool all window embeddings (e.g., mean)
            ch_embeddings = np.stack(ch_embeddings)
            mean_embedding = np.mean(ch_embeddings, axis=0)  # shape: [hidden_dim]
            embeddings.append(mean_embedding)

    return torch.tensor(embeddings, dtype=torch.float32)  # [15, hidden_dim]


In [67]:
channel_embeddings = extract_channel_embeddings(model, data_np, seq_len=50, device='mps')

print("Embeddings shape:", channel_embeddings.shape) 

Embeddings shape: torch.Size([15, 5])


In [408]:
print(channel_embeddings, type(channel_embeddings))

tensor([[ 0.1384,  0.0249, -0.1237,  0.0483,  0.3246],
        [ 0.1396,  0.0263, -0.1283,  0.0463,  0.3248],
        [ 0.1411,  0.0319, -0.1385,  0.0439,  0.3343],
        [ 0.1463,  0.0344, -0.1380,  0.0410,  0.3229],
        [ 0.1486,  0.0395, -0.1465,  0.0364,  0.3263],
        [ 0.1533,  0.0458, -0.1524,  0.0326,  0.3230],
        [ 0.1436,  0.0328, -0.1360,  0.0418,  0.3260],
        [ 0.1234,  0.0047, -0.1002,  0.0658,  0.3327],
        [ 0.1136, -0.0022, -0.0809,  0.0793,  0.3373],
        [ 0.1155,  0.0036, -0.0913,  0.0750,  0.3443],
        [ 0.1314,  0.0233, -0.1253,  0.0569,  0.3434],
        [ 0.1316,  0.0194, -0.1161,  0.0573,  0.3326],
        [ 0.1341,  0.0181, -0.1124,  0.0538,  0.3224],
        [ 0.1338,  0.0187, -0.1141,  0.0541,  0.3250],
        [ 0.1228,  0.0084, -0.1006,  0.0673,  0.3373]]) <class 'torch.Tensor'>


In [414]:
torch.save(channel_embeddings,'/Users/poorvaichandrasen/GCN_IIST/EEG/channel_embeddings.pt')

#### Testing accuracy of RNN on pre-ictal data

In [433]:
preictal_1 = '//Users/poorvaichandrasen/Downloads/Patient_1/Patient_1_preictal_segment_0001.mat'
p_eg_1 = io.loadmat(preictal_1)
preictal_segment_1 = p_eg_1.get('preictal_segment_1')

In [441]:
my_p_data= preictal_segment_1[0][0][0]

In [443]:
data_t = my_p_data[0:15, 280000:280200]
data_t.shape

(15, 200)

In [445]:
SEQ_LEN = 50
data_tf = data_t.copy().astype(np.float32)  # shape (15, 5000)

# Normalize each channel independently
scaler = StandardScaler()
for i in range(data_tf.shape[0]):
    data_tf[i] = scaler.fit_transform(data_tf[i].reshape(-1, 1)).flatten()

In [447]:
# Create (X, y) pairs for next-timestep prediction
X_tf, y_tf = [], []
for ch in data_tf:
    for i in range(len(ch) - SEQ_LEN):
        X_tf.append(ch[i:i+SEQ_LEN])
        y_tf.append(ch[i+SEQ_LEN])

In [457]:
X_tf = np.array(X_tf).reshape(-1, 50, 1)
y_tf = np.array(y_tf).reshape(-1, 1)

In [455]:
evaluate_model(model, torch.from_numpy(X_tf).to(device), torch.from_numpy(y_tf).to(device))

📊 Evaluation Metrics:
✅ Mean Squared Error (MSE):      0.001698
✅ Mean Absolute Error (MAE):     0.029673
✅ R² Score (Explained Variance): 0.9984
Predicted: -0.9123835 Target: -0.9482802 loss = -0.03589672
Predicted: -0.98582804 Target: -1.0475395 loss = -0.06171143
Predicted: -1.1038554 Target: -1.2223217 loss = -0.11846638
Predicted: -1.3064536 Target: -1.3375244 loss = -0.031070828
Predicted: -1.3779618 Target: -1.4324411 loss = -0.05447936
Predicted: -1.4471439 Target: -1.4633759 loss = -0.016232014
Predicted: -1.4635582 Target: -1.4850006 loss = -0.021442413
Predicted: -1.4904213 Target: -1.5291126 loss = -0.038691282
Predicted: -1.5482258 Target: -1.5714993 loss = -0.023273587
Predicted: -1.5768068 Target: -1.6153734 loss = -0.03856659
Predicted: -1.6081676 Target: -1.6232558 loss = -0.015088201
Predicted: -1.6026495 Target: -1.6113578 loss = -0.008708358
Predicted: -1.5883797 Target: -1.580066 loss = 0.008313775
Predicted: -1.5545287 Target: -1.5558535 loss = -0.0013247728
Predi

## Extracting embeddings and saving

In [46]:
parent = '/Users/poorvaichandrasen/Downloads/Patient_1/'
destination = '/Users/poorvaichandrasen/GCN_IIST/EEG/preictal_embeddings/'
n_files = 18
p = 'Patient_1_'


In [52]:
import os

for i in range(n_files):
    mat_path = os.path.join(parent, f"{p}preictal_segment_{str(i+1).zfill(4)}.mat")
    mat_data = io.loadmat(mat_path)

    key = f"preictal_segment_{str(i+1)}"
    segment = mat_data.get(key)

    if segment is None:
        print(f"⚠️ Skipping file: {mat_path} — key {key} not found.")
        continue

    # Extract EEG data
    eeg_data_raw = segment[0][0][0]  # Adjust indexing based on 
    eeg_data = eeg_data_raw[0:15, 280000:280200]

    # Get embeddings
    channel_embeddings = extract_channel_embeddings(model, eeg_data, seq_len=50, device='mps')

    # Save to file
    save_path = os.path.join(destination, f"{p}_embed_{str(i+1).zfill(4)}.pt")
    torch.save(channel_embeddings, save_path)
    
    print(f"{i+1}. Embeddings shape:", channel_embeddings.shape)

/var/folders/3f/pljvlty17172k_x4svc7myr00000gn/T/ipykernel_1847/3422409153.py:32: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:277.)
  return torch.tensor(embeddings, dtype=torch.float32)  # [15, hidden_dim]


1. Embeddings shape: torch.Size([15, 5])
2. Embeddings shape: torch.Size([15, 5])
3. Embeddings shape: torch.Size([15, 5])
4. Embeddings shape: torch.Size([15, 5])
5. Embeddings shape: torch.Size([15, 5])
6. Embeddings shape: torch.Size([15, 5])
7. Embeddings shape: torch.Size([15, 5])
8. Embeddings shape: torch.Size([15, 5])
9. Embeddings shape: torch.Size([15, 5])
10. Embeddings shape: torch.Size([15, 5])
11. Embeddings shape: torch.Size([15, 5])
12. Embeddings shape: torch.Size([15, 5])
13. Embeddings shape: torch.Size([15, 5])
14. Embeddings shape: torch.Size([15, 5])
15. Embeddings shape: torch.Size([15, 5])
16. Embeddings shape: torch.Size([15, 5])
17. Embeddings shape: torch.Size([15, 5])
18. Embeddings shape: torch.Size([15, 5])


In [54]:
destination = '/Users/poorvaichandrasen/GCN_IIST/EEG/interictal_embeddings/'
for i in range(n_files):
    mat_path = os.path.join(parent, f"{p}interictal_segment_{str(i+1).zfill(4)}.mat")
    mat_data = io.loadmat(mat_path)

    key = f"interictal_segment_{str(i+1)}"
    segment = mat_data.get(key)

    if segment is None:
        print(f"⚠️ Skipping file: {mat_path} — key {key} not found.")
        continue

    # Extract EEG data
    eeg_data_raw = segment[0][0][0]  # Adjust indexing based on 
    eeg_data = eeg_data_raw[0:15, 280000:280200]

    # Get embeddings
    channel_embeddings = extract_channel_embeddings(model, eeg_data, seq_len=50, device='mps')

    # Save to file
    save_path = os.path.join(destination, f"{p}i_embed_{str(i+1).zfill(4)}.pt")
    torch.save(channel_embeddings, save_path)
    
    print(f"{i+1}. Embeddings shape:", channel_embeddings.shape)

1. Embeddings shape: torch.Size([15, 5])
2. Embeddings shape: torch.Size([15, 5])
3. Embeddings shape: torch.Size([15, 5])
4. Embeddings shape: torch.Size([15, 5])
5. Embeddings shape: torch.Size([15, 5])
6. Embeddings shape: torch.Size([15, 5])
7. Embeddings shape: torch.Size([15, 5])
8. Embeddings shape: torch.Size([15, 5])
9. Embeddings shape: torch.Size([15, 5])
10. Embeddings shape: torch.Size([15, 5])
11. Embeddings shape: torch.Size([15, 5])
12. Embeddings shape: torch.Size([15, 5])
13. Embeddings shape: torch.Size([15, 5])
14. Embeddings shape: torch.Size([15, 5])
15. Embeddings shape: torch.Size([15, 5])
16. Embeddings shape: torch.Size([15, 5])
17. Embeddings shape: torch.Size([15, 5])
18. Embeddings shape: torch.Size([15, 5])
